In [ ]:
"""
Data preprocessing functions
"""

import pandas as pd
import numpy as np
from config import settings


def load_data(data_path: str = None) -> pd.DataFrame:
    """Load data from Excel file."""
    if data_path is None:
        data_path = settings.DATA_PATH
    
    df = pd.read_excel(data_path, engine='openpyxl')
    df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
    return df


def clean_numeric_column(value):
    """
    Clean numeric columns by converting non-numeric values to NaN.
    
    Args:
        value: Value to clean
        
    Returns:
        Cleaned numeric value or np.nan
    """
    if pd.isna(value):
        return np.nan
    
    # Remove leading/trailing whitespace
    s = str(value).strip()
    
    # Handle values starting with >, <, or special characters
    if s.startswith('>') or s.startswith('＜') or s.startswith('<'):
        s = s[1:]
    
    # Try to convert to float
    try:
        return float(s)
    except ValueError:
        return np.nan


def clean_numeric_data(df: pd.DataFrame) -> pd.DataFrame:
    """Apply cleaning to all numeric columns."""
    df_clean = df.copy()
    for col in settings.FEATURE_COLS:
        df_clean[col] = df_clean[col].apply(clean_numeric_column)
    return df_clean


def remove_all_null_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows where all feature columns are null."""
    check_cols = [col for col in df.columns if col not in 
                  ['ID', 'ID2', 'depart', 'Time', 'Group']]
    df_clean = df.dropna(subset=check_cols, how='all').reset_index(drop=True)
    return df_clean


def get_missingness_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Generate missingness summary statistics."""
    missing_info = pd.DataFrame({
        'n': df.isnull().sum(),
        'percent': df.isnull().sum() / len(df) * 100
    })
    return missing_info